# Google Cloud Platform Workshop - Oefeningen

In dit notebook leer je werken met Google Cloud Platform, specifiek met Vertex AI en de Gemini API.


In [ ]:
import os
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
from google.cloud import storage
from dotenv import load_dotenv
load_dotenv()


---

## Oefening 1: Je eerste Gemini API call

Roep de SDK aan van vertexai.generative_models en vraag iets simpels aan Gemini.

Bijv. "Leg in één zin uit wat cloud computing is"

> **💡 Wat is een SDK?**  
> Een **SDK** (Software Development Kit) is een verzameling tools, bibliotheken en documentatie die ontwikkelaars gebruiken om software te bouwen voor een specifiek platform of dienst—in dit geval de Vertex AI Gemini API.




In [ ]:
# Definieer hier je CONSTANTE_VARIABELEN (bijv. de versie van gemini)
GOOGLE_CLOUD_PROJECT="crypto-triode-477511-g4"
GCP_PROJECT_NAME="vertex-ai-tutorial-project"
GOOGLE_CLOUD_LOCATION="europe-west4"
GEMINI_LOCATION = "europe-west4"
GEMINI_VERSION = "gemini-2.5-flash"
GCP_BUCKET_NAME="bucket_crypto-triode-477511-g4"

In [ ]:
# Initialiseer Vertex AI
vertexai.init(project=GOOGLE_CLOUD_PROJECT, location=GEMINI_LOCATION)

# Initialiseer het generatieve model
model = GenerativeModel(GEMINI_VERSION)

# Generate content
response = model.generate_content("Leg in één zin uit wat cloud computing is", generation_config={"max_output_tokens":1000})

print(response.text)

### 💡 Jouw beurt!

Stel je eigen vraag aan Gemini, met max_output_tokens limiet:

In [ ]:
# Implementeer hier

---

## Oefening 2: Deterministische output

Je kunt de output van Gemini beïnvloeden met parameters zoals temperature en max_output_tokens:

**Opdracht**

Experimenteer met verschillende temperatuur instellingen en zie of je verschil ziet in creativiteit, naarmate je de temperatuur verhoogt:

In [ ]:
config = GenerationConfig(
    temperature=2.0,  # 0.0 = deterministisch, 2.0 = zeer creatief
    max_output_tokens=1000,
)

response = model.generate_content(
    "JOUW_PROMPT",
    generation_config=config
)

print("Output:")
print(response.text)


## Oefening 3: Structured output

Je kunt het model forceren een antwoord te geven in JSON in een bepaald format, door:
1. Expliciet in de prompt te vragen om JSON output met een specifiek schema
2. Een voorbeeld te geven van het gewenste formaat (=*1-shot learning*)

**Opdracht:**

Tweek de prompt totdat je een valide JSON hebt

In [ ]:
# Structured JSON output voorbeeld
json_config = GenerationConfig(
    temperature=0.1,  # Laag voor consistente output
    max_output_tokens=3000,
)

prompt = """
Analyseer de volgende cloud service en geef een gestructureerde output in JSON formaat.

Service: Google Cloud Run

Gebruik EXACT dit JSON schema:
{
  "service_name": "string",
  "category": "string (IaaS/PaaS/SaaS)",
  "pricing_model": "string",
  "primary_use_cases": ["string", "string", "string"],
  "pros": ["string", "string"],
  "cons": ["string", "string"],
  "difficulty_level": "number (1-10)"
}

Geef ALLEEN de JSON output

"""

response = model.generate_content(prompt, generation_config=json_config)
print(response.text)

# Parse de JSON om te verifiëren dat het valide is
import json
try:
    parsed = json.loads(response.text)
    print("\nValide JSON ontvangen!")
    print(f"Service: {parsed['service_name']}")
    print(f"Category: {parsed['category']}")
    print(f"Difficulty: {parsed['difficulty_level']}/10")
except json.JSONDecodeError as e:
    print(f"Geen valide JSON: {e}")

## Oefening 4: Analyseer afbeelding


In [ ]:
import requests
from PIL import Image
from io import BytesIO
from vertexai.generative_models import Part

# Download een afbeelding
image_url = "https://www.google.com/images/branding/googlelogo/2x/googlelogo_color_272x92dp.png"
response = requests.get(image_url)
image = Image.open(BytesIO(response.content))

# Sla de afbeelding op en bekijk hem
image.save("downloaded_image.png")

# Converteer PIL Image naar bytes voor Part.from_image()
img_byte_arr = BytesIO()
image.save(img_byte_arr, format='PNG')
img_byte_arr = img_byte_arr.getvalue()

response_gemini = model.generate_content([
    "Beschrijf deze afbeelding kort. Welk bedrijf of product zie je?",
    Part.from_data(data=img_byte_arr, mime_type="image/png")
])

print(response_gemini.text)

### 💡 Jouw beurt!

Kies je eigen afbeelding en laat Gemini deze analyseren:

In [ ]:
# Jouw code hier - kies een andere image_url
my_image_url = "https://..."


---

## Oefening 5: Data ophalen uit een bucket

In echte cloud applicaties staat data vaak in **Cloud Storage buckets**. Laten we leren hoe we data kunnen ophalen en analyseren.

> **💡 Wat is een Storage Bucket?**  
> Een **Cloud Storage bucket** is een container voor het opslaan van data in de cloud. Je kunt het zien als een map op een server die toegankelijk is via het internet. Perfect voor grote bestanden, backup, en data die je wilt delen tussen services.

We werken met data van temperatuur- en luchtkwaliteitssensoren uit verschillende Nederlandse steden.

In [ ]:
import json

# Define the bucket
bucket = storage.Client().bucket(GCP_BUCKET_NAME)

# Lijst alle bestanden in de bucket
print("Bestanden in bucket:")
for blob in bucket.list_blobs(prefix='data/'):
    print(f"  - {blob.name}")

# Haal een specifiek bestand op
blob = bucket.blob('data/sensor_readings_day1.json')
data_string = blob.download_as_string()

# Parse JSON
sensor_data = json.loads(data_string)

print(f"\n{len(sensor_data)} metingen geladen")
print(f"\nEerste meting:")
print(json.dumps(sensor_data[0], indent=2))

### 💡 Jouw beurt!

Haal nu `sensor_readings_day2.json` op en bekijk de data:

In [ ]:
# Jouw code hier - haal day2 data op

---

## Oefening 6: Data Analyse met Gemini

Nu we data uit de bucket kunnen halen, laten we Gemini deze data analyseren via een prompt

In [ ]:
# Implementeer hier


---

## Oefening 7: Error Handling en Safety Settings

Leer hoe je omgaat met errors en safety filters:

In [ ]:
from vertexai.generative_models import HarmCategory, HarmBlockThreshold

# Configureer safety settings
safety_config = {
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.OFF,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
}

# Error handling
try:
    response = model.generate_content(
        "Wat zijn best practices voor cloud security?",
        safety_settings=safety_config
    )
    print(response.text)
except Exception as e:
    print(f"Error: {type(e).__name__}")
    print(f"Details: {str(e)}")